# 03 - Workflow vs. Agent

## Scenario: Northstar Refund Processing

When Northstar support handles a refund request, the steps are fixed: 
1. **Auth**: Verify the user's identity.
2. **Check Balance**: Ensure the customer is eligible for a refund.
3. **Refund**: Issue the transaction.

Should this be an Agent? **No.** If you give an LLM three tools and say "Process the refund", it might skip the authentication step, or check the balance and then decide to offer a discount instead of a refund. 

When the path is strict, we use a **Deterministic Workflow**. In this module, we will build a strict `langgraph` workflow and compare it to an unpredictable `openai` agent to show why workflows are safer for strict pipelines.

In [1]:
import os
import json
from openai import OpenAI
from typing import TypedDict

# 1. Attempt to use the real API
if os.environ.get("OPENAI_API_KEY"):
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
else:
    # 2. Fallback to our local mock for students without keys
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    import sys
    import os
    sys.path.append(os.path.abspath("../../.."))
    from awsome_agents.mock_openai import MockOpenAI
    client = MockOpenAI()

# 3. Optional: Local LLMs
# If you prefer to use a local model like Llama 3 instead of the mock:
# client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# Let's mock the internal Northstar systems
def auth_user(user_id: str) -> bool:
    print(f"[System] Authenticating {user_id}...")
    return True

def check_balance(user_id: str) -> int:
    print(f"[System] Checking balance for {user_id}...")
    return 50  # Customer has $50 eligible for refund

def issue_refund(user_id: str, amount: int) -> str:
    print(f"🚨 [System] ISSUING REFUND of ${amount} to {user_id}!")
    return "SUCCESS"


## Approach 1: The Predictable LangGraph Workflow

A workflow forces execution down a specific path. The LLM is only used to extract data (like `user_id` and `amount`) from the text, but the execution order is hardcoded.

In [2]:
from pydantic import BaseModel
from langgraph.graph import StateGraph, END

# 1. State
class RefundState(TypedDict):
    ticket_text: str
    user_id: str
    amount_requested: int
    is_authenticated: bool
    balance: int
    status: str

# 2. LLM Parser (Extracts data, does NOT execute tools)
class RefundRequest(BaseModel):
    user_id: str
    amount: int

def node_parse(state: RefundState):
    print("-> Parsing ticket...")
    try:
        completion = client.beta.chat.completions.parse(
            model="gpt-4o",
            messages=[{"role": "user", "content": state["ticket_text"]}],
            response_format=RefundRequest
        )
        data = completion.choices[0].message.parsed
        return {"user_id": data.user_id, "amount_requested": data.amount}
    except Exception:
        # Fallback for mock environments
        return {"user_id": "user_99", "amount_requested": 50}

# 3. Deterministic Nodes
def node_auth(state: RefundState):
    print("-> Forcing Auth check...")
    return {"is_authenticated": auth_user(state["user_id"])}

def node_balance(state: RefundState):
    print("-> Forcing Balance check...")
    return {"balance": check_balance(state["user_id"])}

def node_refund(state: RefundState):
    if state["is_authenticated"] and state["amount_requested"] <= state["balance"]:
        print("-> Executing Refund...")
        issue_refund(state["user_id"], state["amount_requested"])
        return {"status": "Refunded"}
    return {"status": "Denied"}

# 4. Build Strict Graph
builder = StateGraph(RefundState)
builder.add_node("parse", node_parse)
builder.add_node("auth", node_auth)
builder.add_node("balance", node_balance)
builder.add_node("refund", node_refund)

# The edges are hardcoded. The LLM cannot skip steps.
builder.set_entry_point("parse")
builder.add_edge("parse", "auth")
builder.add_edge("auth", "balance")
builder.add_edge("balance", "refund")
builder.add_edge("refund", END)

workflow = builder.compile()

print("--- Running Deterministic Workflow ---")
workflow.invoke({"ticket_text": "I am user_99 and I want my $50 back."})


--- Running Deterministic Workflow ---
-> Parsing ticket...


-> Forcing Auth check...
[System] Authenticating user_99...
-> Forcing Balance check...
[System] Checking balance for user_99...
-> Executing Refund...
🚨 [System] ISSUING REFUND of $50 to user_99!


{'ticket_text': 'I am user_99 and I want my $50 back.',
 'user_id': 'user_99',
 'amount_requested': 50,
 'is_authenticated': True,
 'balance': 50,
 'status': 'Refunded'}

## Approach 2: The Unpredictable Agent

If we give an agent the same tools (`auth_user`, `check_balance`, `issue_refund`), it *might* do them in order. Or it might not. It might decide to skip authentication because it's "confident" it knows the user. Or it might loop endlessly.

In [3]:
# We simulate an unpredictable LLM by purposefully omitting the auth check
# in our mock to demonstrate what happens when you trust an agent with a strict pipeline.

def simulated_unpredictable_agent(ticket_text: str):
    print("--- Running Unpredictable Agent ---")
    print("Agent Thought: I see the user wants $50. I have the issue_refund tool.")
    
    # The agent skipped auth_user and check_balance!
    print("Agent Thought: I will just issue the refund directly to save time.")
    issue_refund("user_99", 50)
    
    print("Agent Final Answer: I have refunded your $50.")

simulated_unpredictable_agent("I am user_99 and I want my $50 back.")


--- Running Unpredictable Agent ---
Agent Thought: I see the user wants $50. I have the issue_refund tool.
Agent Thought: I will just issue the refund directly to save time.
🚨 [System] ISSUING REFUND of $50 to user_99!
Agent Final Answer: I have refunded your $50.


## Watch For

- **Agentic Hammer**: When you learn how to build agents, everything looks like a nail. But 80% of enterprise processes are actually just strict workflows.
- **Lost Compliance**: If SOC2 compliance requires that every refund has an auth check, you cannot use an agent. You must use a workflow where the auth edge is hardcoded.

## Checkpoint

**1. When should you choose a Workflow over an Agent?**
- A) When the task requires creative problem solving and dynamic tool usage.
- B) When the execution path is strict, compliance is required, and steps cannot be skipped.
- C) When you want to save money on API keys.
- D) When the task requires web browsing.

**2. In the LangGraph workflow above, who decides what node runs next?**
- A) The LLM.
- B) The user.
- C) The hardcoded edges (e.g. `builder.add_edge("auth", "balance")`).
- D) The system prompt.
